<a href="https://colab.research.google.com/github/Griffindor458/MLprojects/blob/main/LLM_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input,decode_predictions
import numpy as np


In [4]:
from tensorflow.keras.preprocessing import image

In [21]:
model = ResNet50(weights = 'imagenet')

In [22]:
img_path = '/content/tomato.png'

In [23]:
img = image.load_img(img_path, target_size =(224,224))

In [24]:
x = image.img_to_array(img)
x = np.expand_dims(x, axis =0)
x = preprocess_input(x)

In [25]:
preds = model.predict(x)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


In [26]:
preds

array([[1.09310395e-06, 2.80831391e-05, 5.27540010e-07, 1.03207174e-06,
        1.38879761e-06, 4.20836732e-05, 1.10062717e-06, 4.22522935e-05,
        3.07822302e-05, 2.69083139e-05, 8.79621712e-06, 2.70516284e-05,
        8.62876550e-06, 7.00337796e-06, 2.97911502e-05, 1.17827685e-05,
        1.05313084e-05, 1.48062845e-05, 1.72598593e-05, 1.89672428e-05,
        2.35493849e-06, 5.34878745e-06, 5.69209897e-06, 1.04627052e-05,
        1.60012974e-06, 3.73720491e-06, 2.75962384e-06, 5.51984231e-05,
        7.21449305e-06, 4.26966653e-06, 2.92391292e-07, 1.43603052e-06,
        1.26188002e-06, 4.18988577e-07, 3.81968721e-06, 2.83435497e-06,
        6.52359086e-06, 5.84515510e-06, 1.51772083e-05, 1.52850953e-06,
        8.68002735e-06, 3.74208576e-05, 3.22782284e-06, 2.09914288e-06,
        5.84754116e-06, 2.20704314e-05, 5.85943189e-06, 2.42924034e-06,
        4.17212618e-07, 5.62499793e-07, 9.28326585e-07, 3.57900099e-05,
        8.27994518e-05, 1.49937172e-04, 2.09680493e-05, 4.191818

In [27]:
for _, label , prob in decode_predictions(preds , top = 10)[0]:
  print(f"{label}:{prob:.4f}")

pitcher:0.1894
orange:0.0903
ping-pong_ball:0.0395
vase:0.0387
tray:0.0320
mask:0.0319
perfume:0.0285
caldron:0.0241
sunscreen:0.0221
piggy_bank:0.0216


In [29]:
from transformers import BertTokenizer ,BertForMaskedLM
import torch
import torch.nn.functional as F

In [31]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [32]:
text = "the future of IT industry is [MASK]."

In [33]:
inputs = tokenizer(text ,return_tensors="pt")

In [34]:
with torch.inference_mode():
  logits = model(**inputs).logits

In [35]:
mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple = True)[1]

In [36]:
probs = F.softmax(logits[0,mask_idx], dim=-1)
top_k = torch.topk(probs , k =5,dim= -1)

In [37]:
print("Top-5 predictions for [MASK]: ")
for token_id , prob in zip(top_k.indices[0],top_k.values[0]):
  token = tokenizer.decode([token_id])
  print(f"{token:<12} -> {prob.item():.4f}")

Top-5 predictions for [MASK]: 
uncertain    -> 0.7427
unknown      -> 0.0487
unclear      -> 0.0376
clear        -> 0.0095
promising    -> 0.0076


In [38]:
from transformers import pipeline ,AutoModelForSequenceClassification , AutoTokenizer

model_name = "textattack/bert-base-uncased-SST-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)


classifier = pipeline("sentiment-analysis" , model = model , tokenizer = tokenizer)

result = classifier("The new GPT 5 is getting celebrated !")

print(result)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'LABEL_1', 'score': 0.9990511536598206}]


In [39]:
from transformers import BertTokenizer
tokn = BertTokenizer.from_pretrained("bert-base-uncased")
output = tokn("GenAI is Future",return_tensors = "pt")
print(output)

{'input_ids': tensor([[ 101, 8991, 4886, 2003, 2925,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}
